# ToR Review Assistant
This notebook implements the ToR Review Assistant. The system **combines a generated checklist with LLM-based classification, reasoning, and summarisation** to support the review of draft evaluation ToRs. The main steps of the pipeline are the following:
*   Load the checklist and individually (de)activate items based on the needs,
*   Parsing a draft ToR (including section detection),
*   Checking fail signals including identfying source paragraph and rationale for why (not) triggering a signal,
*   Providing revision suggestions for fail signals. Add a text proposition only if useful and if enough evidence is available,
*   Producing a summary report of the revision, the ToR Review Report.



## Installations, imports and paths
At the beginning, the required libraries are installed, the necessary modules imported, and the input and output paths defined, which are used throughout the ToR Review Assistant pipeline.

In [ ]:
# Required libraries installation and downloads
!pip install spacy
!python -m spacy download xx_ent_wiki_sm
!pip install python-docx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 95.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('xx_ent_wiki_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 16.0 MB/s eta 0:00:00


In [ ]:
# Import libraries
import pandas as pd
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Pt, RGBColor
import json
import os
import re
import zipfile
import xml.etree.ElementTree as ET
import spacy
from docx.shared import Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH

nlp = spacy.load("xx_ent_wiki_sm")
NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
WNS = "{%s}" % NS["w"]

In [ ]:
# Import Colab drive module and mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Project paths and configuration

# Project directory
PROJECT_ROOT = "/content/drive/MyDrive/HSLU/Masters_Thesis"

# API key
ENV_PATH = f"{PROJECT_ROOT}/.env"

# Input - Checklist
CHECKLIST_PATH = f"{PROJECT_ROOT}/Checklist_Documents/Final_Checklist.csv"

# Input - ToR documents
INPUT_DIR = f"{PROJECT_ROOT}/Input"

TOR_FILES = {
    "example_1": f"{INPUT_DIR}/37_Test_FSDRP_II.docx",
    "example_2": f"{INPUT_DIR}/19_240919_ToR Impact Evaluation PARIeD.docx",
    "example_3": f"{INPUT_DIR}/28_Terms of Reference Developmental Evaluation.docx",
    "example_4": f"{INPUT_DIR}/48_ToR RCDN+ Evaluation v0.3 Draft_mcn.docx",
    "example_5": f"{INPUT_DIR}/2_Final ToR Ex-Post Susatainble Cocoa Production Program.docx",
    "example_6": f"{INPUT_DIR}/46_SFF_External Evaluation ToR_May 2025_mcn v2.docx",
    "example_7": f"{INPUT_DIR}/40_ToR Evaluation EEA Projects in Albania and Serbia_clean.docx",
    "example_8": f"{INPUT_DIR}/1_240207_ToR_external project evaluation_RESD_draft.docx",
    "example_13": f"{INPUT_DIR}/13_ToRs Mid-Term Evaluation SLPI Phase I FINAL.docx",
    "example_14": f"{INPUT_DIR}/Task_1_ToR_Competitiveness_Programme.docx",
    "example_15": f"{INPUT_DIR}/Task_2_Sustainable_Tourism.docx"
}

# Select a ToR for review
SELECTED_TOR = "example_15"
TOR_INPUT_PATH = TOR_FILES[SELECTED_TOR]

# SECO Logo
LOGO_PATH = f"{PROJECT_ROOT}/Input/Logo.jpg"

# Output directory
OUTPUT_DIR = f"{PROJECT_ROOT}/Output"
REVISED_TOR_DIR = f"{OUTPUT_DIR}/Revised_ToR"

# Cache / intermediate files
CACHE_DIR = f"{PROJECT_ROOT}/Cache"

# Logging
LOG_DIR = f"{PROJECT_ROOT}/logs"

## Load Checklist
As a first step, we load and prepare the checklist used in the ToR review process. Some variables are changed from numeric codes into words to make them easier to understand, and a few basic checks are carried out to make sure the data was imported correctly and is ready for further analysis.

In [ ]:
# Check file exists
checklist_file = Path(CHECKLIST_PATH)
print(f"Checklist file exists: {checklist_file.exists()}")
print(f"Checklist path: {checklist_file}")

# Load CSV
checklist_df = pd.read_csv(CHECKLIST_PATH, encoding="latin1")

checklist_df = checklist_df.rename(columns={
    "fix_mode (0 = instruction-only, 1 = instruction plus suggestion)": "fix_mode"
})

# Convert values
checklist_df["fix_mode"] = checklist_df["fix_mode"].map({
    0: "instruction_only",
    1: "instruction_plus_suggestion"
})

checklist_df = checklist_df.rename(columns={
    "importance (0 = low, 1 = medium, 2 = high)": "importance"
})

checklist_df["importance"] = checklist_df["importance"].map({
    0: "low",
    1: "medium",
    2: "high"
})

# Basic checks
print("\nShape:")
print(checklist_df.shape)

print("\nColumns and first 5 rows:")
display(checklist_df.head())

Checklist file exists: True
Checklist path: /content/drive/MyDrive/HSLU/Masters_Thesis/Checklist_Documents/Final_Checklist.csv

Shape:
(93, 10)

Columns and first 5 rows:


,ToR Section,Dimension,Requirement,fail_signal_id,fail_signal,fix_pattern_id,fix_pattern,fix_mode,importance,"activated (1 = yes, 0 = no)"
0,1. Purpose of the Evaluation,Purpose,The ToR must explicitly state the primary purp...,1.0,Multiple evaluation purposes are mentioned but...,1.0,State the primary purpose of the evaluation.,instruction_only,high,1.0
1,1. Purpose of the Evaluation,Purpose,The ToR must explicitly state the primary purp...,2.0,No evaluation purpose is stated.,2.0,Specify whether the evaluation is primarily fo...,instruction_only,high,1.0
2,1. Purpose of the Evaluation,Priorisation,If the ToR lists more than one evaluation purp...,3.0,If the ToR lists more than one evaluation purp...,3.0,Clarify the relative priority of the different...,instruction_only,high,1.0
3,1. Purpose of the Evaluation,Decision-use link,The ToR must explain which decisions or action...,4.0,The ToR does not specify which decisions or ac...,4.0,State which decisions or actions the evaluatio...,instruction_only,high,1.0
4,1. Purpose of the Evaluation,User identification,The ToR must identify the primary intended use...,5.0,The intended users or audiences of the evaluat...,5.0,Specify the primary intended users of the eval...,instruction_only,high,1.0


We display a few rows to manually verify that the checklist items were loaded correctly.

In [ ]:
# Display specific rows by index
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

display(checklist_df.iloc[[16, 48, 86]])

,ToR Section,Dimension,Requirement,fail_signal_id,fail_signal,fix_pattern_id,fix_pattern,fix_mode,importance,"activated (1 = yes, 0 = no)"
16,2. Description of the Intervention,Implementation structure,"The ToR must include a succinct summary of the projectÕs implementation structure (e.g., implementing organisations, governance structure, coordination arrangements).",17.0,No summary of the projectÕs implementation structure is provided.,17.0,Summarise the projectÕs implementation structure clearly and concisely.,instruction_only,medium,1.0
48,4. Evaluation Methods and Process,Sampling,"If sampling or case selection is required to answer evaluation questions, the ToR must specify the sampling approach and representation requirements, or justify purposive deep-dives.",49.0,"Evaluation questions require sampling or case selection, but no sampling approach, representation requirements, or case selection guidance is provided.",49.0,"Specify the sampling approach, representation requirements, or case selection criteria.",instruction_only,medium,1.0
86,8. Budget,Effort,The ToR must specify the expected level of effort as a number of working days aligned with the scope and deliverables.,87.0,"The ToR provides no estimate of working days, provides only a single total without breakdown, or provides estimates inconsistent with the scope.",87.0,"Provide a clear estimate of working days, with a breakdown aligned to the scope.",instruction_only,medium,1.0


## (De)activate Items
Next, we can **manually activate or deactivate checklist items** depending on the use case. For example, in a high-stakes evaluation all items may be activated, while for a quick review only selected items may be used to keep the process efficient and fast. In this example, we deactivate 28 items.

In [ ]:
# (De)activate items

ids_to_deactivate = [3, 7, 9, 11, 13, 16, 18, 24, 25, 26, 28, 31, 39, 44, 49, 50, 51, 52, 54, 55, 58, 68, 72, 76, 77, 78, 83, 88]

checklist_df.loc[
    checklist_df["fail_signal_id"].isin(ids_to_deactivate),
    "activated (1 = yes, 0 = no)"
] = 0

In [ ]:
# Count activated and deactivated rows
col = "activated (1 = yes, 0 = no)"

activated = checklist_df[col].eq(1).sum()
deactivated = checklist_df[col].eq(0).sum()

print("Activated:", activated)
print("Deactivated:", deactivated)

Activated: 64
Deactivated: 28


Next, we create a new **dataset containing only the activated fail signals** that will be used in the review process. We also keep only the relevant columns needed for the later analysis and display a short preview to verify the result.

In [ ]:
# Build activated fail-signal dataset
col = "activated (1 = yes, 0 = no)"

fail_signals_df = (
    checklist_df.loc[
        checklist_df[col].eq(1),
        ["ToR Section", "fail_signal_id", "fail_signal", "fix_pattern", "fix_mode", "importance"]
    ]
    .rename(columns={"ToR Section": "chapter_text"})
    .copy()
)

print(f"Activated fail signals: {len(fail_signals_df)}")
display(fail_signals_df.head())

Activated fail signals: 64


,chapter_text,fail_signal_id,fail_signal,fix_pattern,fix_mode,importance
0,1. Purpose of the Evaluation,1.0,Multiple evaluation purposes are mentioned but no primary purpose or equal weighting is explicitly stated.,State the primary purpose of the evaluation.,instruction_only,high
1,1. Purpose of the Evaluation,2.0,No evaluation purpose is stated.,"Specify whether the evaluation is primarily for learning, accountability, or decision-making.",instruction_only,high
3,1. Purpose of the Evaluation,4.0,The ToR does not specify which decisions or actions will be informed by the evaluation findings.,State which decisions or actions the evaluation findings are expected to inform.,instruction_only,high
4,1. Purpose of the Evaluation,5.0,The intended users or audiences of the evaluation are not mentioned.,Specify the primary intended users of the evaluation findings.,instruction_only,high
5,1. Purpose of the Evaluation,6.0,User groups or audiences are mentioned but their expected use of the evaluation findings is not described.,Linking each main user group to how they will use the evaluation findings.,instruction_only,high


## 4.2.1 Parsing Draft ToRs
In this step, we parse the draft ToR document and convert it into a structured format for reviewing. The parser identifies paragraphs, headings, and sections, while also cleaning and organising the extracted text. We first check whether the selected draft ToR file is available and can be accessed.

In [ ]:
# Verify selected ToR input file
print("Selected ToR key:", SELECTED_TOR)
print("Resolved path:", TOR_INPUT_PATH)
print("Exists:", os.path.exists(TOR_INPUT_PATH))

tor_file = Path(TOR_INPUT_PATH)
print("Suffix:", tor_file.suffix)
print("Size (bytes):", tor_file.stat().st_size if tor_file.exists() else "File not found")

Selected ToR key: example_15
Resolved path: /content/drive/MyDrive/HSLU/Masters_Thesis/Input/Task_2_Sustainable_Tourism.docx
Exists: True
Suffix: .docx
Size (bytes): 181580


Then, we define the **SECO ToR sections together with keywords** that help assign parsed text to the correct section.

In [ ]:
# SECO buckets and keyword mapping

SECO_BUCKETS = [
    "1. Purpose of the Evaluation",
    "2. Description of the Intervention",
    "3. Scope and Focus of the Evaluation / Evaluation Questions",
    "4. Evaluation Methods and Process",
    "5. Deliverables",
    "6. Schedule",
    "7. Evaluation Team / Qualifications",
    "8. Budget",
    "9. Annexes",
]

UNKNOWN_BUCKET = "(unknown)"

BUCKET_KEYWORDS = {
    "1. Purpose of the Evaluation": [
        "purpose","objective","aim","goal","intended use","intended user","learning","accountability"
    ],
    "2. Description of the Intervention": [
        "intervention","programme","program","project","background","context","stakeholder","theory of change","logframe"
    ],
    "3. Scope and Focus of the Evaluation / Evaluation Questions": [
        "scope","focus","evaluation question","questions","criteria","dac","oecd"
    ],
    "4. Evaluation Methods and Process": [
        "method","methodology","approach","design","sampling","data collection","analysis","triangulation"
    ],
    "5. Deliverables": [
        "deliverable","report","inception report","draft report","final report","presentation"
    ],
    "6. Schedule": [
        "schedule","timeline","workplan","milestone","deadline","duration"
    ],
    "7. Evaluation Team / Qualifications": [
        "team","qualifications","expertise","experience","consultant","evaluator"
    ],
    "8. Budget": [
        "budget","cost","fees","financial","rate","days","expenses"
    ],
    "9. Annexes": [
        "annex","appendix","attachments"
    ],
}

### Helper Functions
We define **helper functions** that identify headings, clean heading text, detect annexes and front matter, and assign headings to the correct SECO section.

In [ ]:
# Define heading styles that should be treated as section headings
HEADING_STYLE_IDS = {
    "seco-h1", "seco-h1-sub",
    "heading1", "heading2", "heading3",
    "überschrift1", "überschrift2", "überschrift3",
    "uberschrift1", "uberschrift2", "uberschrift3",
    "berschrift1", "berschrift2", "berschrift3"
}

# Extract text from a paragraph and mark images if present
def extract_paragraph_text(p):
    texts = []
    texts.extend((t.text or "") for t in p.findall(".//w:t", NS))
    if p.findall(".//w:drawing", NS) or p.findall(".//w:pict", NS):
        texts.append("[IMAGE]")
    return " ".join(texts).strip()

# Get the paragraph style ID from the Word document
def get_style_id(p):
    ps = p.find("./w:pPr/w:pStyle", NS)
    return ps.get(WNS + "val") if ps is not None else ""

# Standardise style IDs for consistent matching
def normalize_style_id(style_id):
    return (style_id or "").strip().lower()

# Read the outline level of a heading if available
def get_outline_level(p):
    ol = p.find("./w:pPr/w:outlineLvl", NS)
    if ol is not None and ol.get(WNS + "val") is not None:
        return int(ol.get(WNS + "val"))
    return None

# Detect headings based on numbering patterns
def looks_like_heading(text):
    t = normalize_heading_text(text)
    if not t or len(t) > 150:
        return False
    return bool(re.match(r"^\d+(\.\d+)*\s*[A-Z]", t))

# Detect annex or appendix headings
def is_annex_heading(text):
    t = normalize_heading_text(text)
    return bool(re.match(r"^(ANNEX|Annex|APPENDIX|Appendix)\s*\d*", t)) and len(t.split()) <= 12

# Assign headings to the most suitable SECO section
def map_heading_to_seco_bucket(text):
    t = normalize_heading_text(text).lower()

    if is_annex_heading(text):
        return "9. Annexes"

    scores = {
        b: sum(1 for k in BUCKET_KEYWORDS[b] if k in t)
        for b in SECO_BUCKETS
    }

    best_bucket, best_score = max(scores.items(), key=lambda kv: kv[1])
    return best_bucket if best_score > 0 else UNKNOWN_BUCKET

# Estimate heading hierarchy level
def infer_heading_level(text, outline_level, style_id):
    style_id = normalize_style_id(style_id)

    if is_annex_heading(text):
        return 1

    if outline_level is not None:
        return outline_level + 1

    if style_id in {"seco-h1", "heading1", "überschrift1", "uberschrift1", "berschrift1"}:
        return 1
    if style_id in {"seco-h1-sub", "heading2", "überschrift2", "uberschrift2", "berschrift2"}:
        return 2
    if style_id in {"heading3", "überschrift3", "uberschrift3", "berschrift3"}:
        return 3

    m = re.match(r"^(\d+(?:\.\d+)*)", (text or "").strip())
    if m:
        return len(m.group(1).split("."))

    return None

# Decide whether a paragraph should be treated as a heading
def is_heading_candidate(text, style_id, outline_level):
    style_id = normalize_style_id(style_id)
    return (
        style_id in HEADING_STYLE_IDS
        or outline_level is not None
        or looks_like_heading(text)
        or is_annex_heading(text)
    )

# Identify paragraphs that likely contain body text
def is_body_text(text):
    t = (text or "").strip()
    return len(t.split()) >= 8

# Clean and standardise heading text
def normalize_heading_text(text):
    t = (text or "").strip()

    # Fix broken spacing inside words
    t = re.sub(r"\b([A-Za-z])\s+([A-Za-z]{2,})\b", r"\1\2", t)

    # Remove repeated spaces
    t = re.sub(r"\s+", " ", t).strip()

    return t

# Detect front matter such as tables of contents or title pages
def is_front_matter_text(text):
    t = normalize_heading_text(text).lower()

    front_matter_patterns = [
        r"^list of contents$",
        r"^content:?$",
        r"^table of contents$",
        r"^request for proposals$",
        r"^prepared by:?$",
        r"^document version:?$",
        r"^template version:?$",
        r"^date:?$",
        r"^part [ivx\d]+",
    ]

    return any(re.match(p, t) for p in front_matter_patterns)

This function reads the draft ToR document and extracts its text, headings, tables, and structure from the .docx file. It then **organises the content into sections and assigns each paragraph to the most relevant SECO category** for later analysis.

In [ ]:
def parse_tor_docx(docx_path):
    # Store extracted paragraphs and table rows
    raw_rows = []
    para_id = 0

    # Open the .docx file
    with zipfile.ZipFile(docx_path) as z:
        root = ET.fromstring(z.read("word/document.xml"))
        body = root.find(".//w:body", NS)

        # Loop through all document elements
        for elem in body:

            # Process normal paragraphs
            if elem.tag == f"{WNS}p":
                p = elem
                text = extract_paragraph_text(p)

                # Skip empty paragraphs
                if not text:
                    continue

                # Extract formatting and heading information
                style_id = get_style_id(p)
                outline_level = get_outline_level(p)
                is_heading = is_heading_candidate(text, style_id, outline_level)
                heading_level = infer_heading_level(text, outline_level, style_id) if is_heading else None

                # Store paragraph information
                raw_rows.append({
                    "paragraph_id": para_id,
                    "paragraph_text": text,
                    "is_heading": is_heading,
                    "heading_level": heading_level,
                    "style_id": style_id,
                })

                para_id += 1

            # Process tables separately
            elif elem.tag == f"{WNS}tbl":
                tbl = elem

                # Loop through table rows
                for row in tbl.findall(".//w:tr", NS):
                    cells = []

                    # Extract text from each table cell
                    for cell in row.findall(".//w:tc", NS):
                        cell_text = " ".join((t.text or "") for t in cell.findall(".//w:t", NS)).strip()

                        if cell_text:
                            cells.append(cell_text)

                    # Skip empty table rows
                    if not cells:
                        continue

                    # Store table row as one text entry
                    raw_rows.append({
                        "paragraph_id": para_id,
                        "paragraph_text": " | ".join(cells),
                        "is_heading": False,
                        "heading_level": None,
                        "style_id": "TABLE",
                    })

                    para_id += 1

    # Convert extracted content into a dataframe
    df = pd.DataFrame(raw_rows)

    # Identify the true start of the ToR
    # Ignore front matter such as title pages or tables of contents
    start_idx = None

    for i in df.index:
        row = df.loc[i]

        if row["is_heading"] and not is_front_matter_text(row["paragraph_text"]):

            # Check whether real body text follows the heading
            window = df.loc[i+1:i+4, "paragraph_text"].tolist()

            if any(is_body_text(t) for t in window):
                start_idx = i
                break

    # Fallback to the beginning if no start is detected
    if start_idx is None:
        start_idx = 0

    # Keep only the relevant document content
    df = df.loc[start_idx:].reset_index(drop=True)

    # Rebuild the heading hierarchy
    rows = []
    heading_stack = {}

    for _, row in df.iterrows():
        text = row["paragraph_text"]
        is_heading = row["is_heading"]
        heading_level = row["heading_level"]

        # Update heading hierarchy when a new heading appears
        if is_heading and heading_level is not None:
            heading_stack[heading_level] = text

            # Remove lower-level headings when moving up the hierarchy
            for lvl in list(heading_stack.keys()):
                if lvl > heading_level:
                    del heading_stack[lvl]

        # Create a full heading path
        heading_path = " > ".join(
            heading_stack[lvl] for lvl in sorted(heading_stack.keys())
        ) if heading_stack else "(no heading)"

        # Assign content to the most relevant SECO section
        chapter_text = map_heading_to_seco_bucket(
            heading_stack.get(1, "(no chapter)")
        )

        # Store final structured output
        rows.append({
            "paragraph_id": row["paragraph_id"],
            "paragraph_text": text,
            "is_heading": is_heading,
            "heading_level": heading_level,
            "heading_path": heading_path,
            "chapter_text": chapter_text,
            "style_id": row["style_id"],
        })

    # Return the structured parsed ToR
    return pd.DataFrame(rows)

### Run Parser
The parser processes the draft ToR and converts it into a dataframe containing the extracted text and document structure. In this example, the parsed output contains 201 rows and 7 columns with information such as paragraph text, headings, and assigned sections.

In [ ]:
# Run parser
parsed_tor_df = parse_tor_docx(TOR_INPUT_PATH)

# Display settings
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Show overall dataframe size
print("Shape:", parsed_tor_df.shape)

# Display 5 random parsing results as examples
print("\nExample parsing results:")
display(
    parsed_tor_df[
        [
            "paragraph_id",
            "paragraph_text",
            "is_heading",
            "heading_level",
            "heading_path",
            "chapter_text"
        ]
    ].sample(5, random_state=42)
)

Shape: (201, 7)

Example parsing results:


,paragraph_id,paragraph_text,is_heading,heading_level,heading_path,chapter_text
95,111,How to use this assessment grid :,False,NaN,Budget,8. Budget
15,31,History and context of the intervention,False,NaN,Description of the Intervention,2. Description of the Intervention
30,46,How d oes the Programme help to develop framework conditions conducive to the development of sustainable tourism and what is the role of the S TAP and SDR Atlas bringing together key actors involved in the development of tourism ?,False,NaN,Scope and Focus of the Evaluation / Evaluation Questions,3. Scope and Focus of the Evaluation / Evaluation Questions
158,174,Executive summary (max. 2 pages),False,NaN,Annex 2 : C ontent and structure of the report,9. Annexes
128,144,If an additional sub-criteria is relevant please formulate it here | select | Click here to enter text.,False,NaN,Budget,8. Budget


### Cleaning and Anonymising
Here, we clean formatting and spacing issues in the parsed text that may result from document extraction. We anonymise sensitive information in the parsed text by replacing email addresses and detected person names with placeholders. This helps protect privacy before the text is used in later analysis steps and sent to an LLM.

In [ ]:
# Clean spacing artifacts in parsed text
def clean_spacing(text):
    if pd.isna(text):
        return text

    s = str(text)

    # collapse repeated whitespace
    s = re.sub(r"\s+", " ", s)

    # remove spaces before punctuation
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)

    # remove spaces after opening brackets / before closing brackets
    s = re.sub(r"([(\[]) +", r"\1", s)
    s = re.sub(r" +([)\]])", r"\1", s)

    # fix spaced years / numbers like "202 4" -> "2024"
    s = re.sub(r"(?<=\d)\s+(?=\d)", "", s)

    # fix broken all-caps acronyms like "F S DRP2" -> "FSDRP2"
    s = re.sub(r"\b(?:[A-Z]\s+){2,}[A-Z0-9]+\b", lambda m: m.group(0).replace(" ", ""), s)

    # trim
    s = s.strip()

    return s

parsed_tor_df["paragraph_text"] = parsed_tor_df["paragraph_text"].apply(clean_spacing)

In [ ]:
# Email anonymization
EMAIL_RE = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")

def anonymize_email(text):
    if pd.isna(text):
        return text
    return EMAIL_RE.sub("[EMAIL]", str(text))

# Person anonymization with spaCy if already loaded
def anonymize_persons(text):
    if pd.isna(text):
        return text

    s = str(text)

    # fast path: skip if no letters
    if not re.search(r"[A-Za-z]", s):
        return s

    doc = nlp(s)
    chars = list(s)

    for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
        if ent.label_ in {"PER", "PERSON"} and len(ent.text.split()) >= 2:
            chars[ent.start_char:ent.end_char] = list("[PERSON]")

    return "".join(chars)

# Apply only to paragraph text
parsed_tor_df["paragraph_text"] = parsed_tor_df["paragraph_text"].apply(anonymize_email)
parsed_tor_df["paragraph_text"] = parsed_tor_df["paragraph_text"].apply(anonymize_persons)

### Aggregation and Check
We aggregate the identified paragraphs into their respective sections to create a section-level view of the ToR. We then display previews of the extracted content, calculate basic section statistics, and highlight possible parsing issues for the user.

In [ ]:
# Aggregate per section
chapter_texts_df = (
    parsed_tor_df
    .groupby("chapter_text", dropna=False)["paragraph_text"]
    .apply(lambda x: "\n\n".join(x.astype(str)))
    .reset_index()
    .rename(columns={"paragraph_text": "chapter_full_text"})
)

# Display one random aggregated chapter as example
display(chapter_texts_df.sample(1, random_state=42))

chapter_text  \
8    8. Budget   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

In [ ]:
# High-level summary
file_name = os.path.basename(TOR_INPUT_PATH)
total_paragraphs = len(parsed_tor_df)

sections_found = [
    s for s in SECO_BUCKETS
    if s in set(parsed_tor_df["chapter_text"].dropna())
]
n_sections_found = len(sections_found)

print("=== ToR Upload Summary ===")
print(f"File: {file_name}")
print(f"Paragraphs parsed: {total_paragraphs}")
print(f"Sections detected: {n_sections_found} / {len(SECO_BUCKETS)}")

print("\nSections found:")
for s in SECO_BUCKETS:
    mark = "✔" if s in sections_found else "✖"
    print(f"{mark} {s}")

# Section preview
print("\n=== Section Preview ===")

chapter_texts_df = (
    parsed_tor_df
    .groupby("chapter_text", dropna=False)["paragraph_text"]
    .apply(lambda x: "\n\n".join(x.astype(str)))
    .reset_index()
    .rename(columns={"paragraph_text": "chapter_full_text"})
)

for _, row in chapter_texts_df.iterrows():
    chapter = row["chapter_text"]
    if chapter == UNKNOWN_BUCKET:
        continue

    full_text = str(row["chapter_full_text"]).strip()
    preview = full_text[:350].replace("\n", " ")
    if len(full_text) > 350:
        preview += "..."

    print(f"\n{chapter}")
    print(preview)


# Basic stats per section
section_stats_df = (
    parsed_tor_df[parsed_tor_df["chapter_text"] != UNKNOWN_BUCKET]
    .groupby("chapter_text", dropna=False)
    .agg(
        paragraphs=("paragraph_text", "count"),
        words=("paragraph_text", lambda x: sum(len(str(t).split()) for t in x))
    )
    .reset_index()
    .rename(columns={"chapter_text": "section"})
)

print("\n=== Section Statistics ===")
display(section_stats_df)


# Data quality checks
unknown_count = (parsed_tor_df["chapter_text"] == UNKNOWN_BUCKET).sum()
heading_missing_level = (
    (parsed_tor_df["is_heading"] == True) &
    (parsed_tor_df["heading_level"].isna())
).sum()

duplicate_section_headings = (
    parsed_tor_df.loc[parsed_tor_df["is_heading"] == True, "chapter_text"]
    .value_counts()
)
duplicate_section_headings = duplicate_section_headings[
    (duplicate_section_headings.index != UNKNOWN_BUCKET) &
    (duplicate_section_headings > 1)
]

print("\n=== Data Quality Notes ===")
if unknown_count == 0 and heading_missing_level == 0 and len(duplicate_section_headings) == 0:
    print("No obvious parsing issues detected.")
else:
    if unknown_count > 0:
        print(f"- {unknown_count} paragraphs assigned to {UNKNOWN_BUCKET}")
    if heading_missing_level > 0:
        print(f"- {heading_missing_level} headings detected without heading level")
    if len(duplicate_section_headings) > 0:
        print("- Sections appearing multiple times as headings:")
        for section, count in duplicate_section_headings.items():
            print(f"  • {section}: {count} heading occurrences")

=== ToR Upload Summary ===
File: Task_2_Sustainable_Tourism.docx
Paragraphs parsed: 201
Sections detected: 9 / 9

Sections found:
✔ 1. Purpose of the Evaluation
✔ 2. Description of the Intervention
✔ 3. Scope and Focus of the Evaluation / Evaluation Questions
✔ 4. Evaluation Methods and Process
✔ 5. Deliverables
✔ 6. Schedule
✔ 7. Evaluation Team / Qualifications
✔ 8. Budget
✔ 9. Annexes

=== Section Preview ===

1. Purpose of the Evaluation
Purpose of the Evaluation  The S TSM Programme is scheduled to end in December 2024. SECO, as the donor of this program me, commissions an external evaluation in the last year of implementation with the following main objectives:  S trategically evaluate the relevance and effectiveness of the design and programme set-up for the implementation of [P...

2. Description of the Intervention
Description of the Intervention  The overall project budget is CHF 3 ’ 850 ’000. SECO contributes CHF 3 ’ 500’000 and the implementing agency is Swisscontact.  Gene

,section,paragraphs,words
0,1. Purpose of the Evaluation,6,119
1,2. Description of the Intervention,12,511
2,3. Scope and Focus of the Evaluation / Evaluation Questions,22,449
3,4. Evaluation Methods and Process,10,165
4,5. Deliverables,7,142
5,6. Schedule,12,179
6,7. Evaluation Team / Qualifications,18,261
7,8. Budget,65,2872
8,9. Annexes,47,343



=== Data Quality Notes ===
- 2 paragraphs assigned to (unknown)
- Sections appearing multiple times as headings:
  • 9. Annexes: 5 heading occurrences
  • 5. Deliverables: 2 heading occurrences


In [ ]:
# Build full ToR context

full_tor_context = "\n".join(
    f"[PID:{row.paragraph_id}] [SECTION:{row.chapter_text}] {row.paragraph_text}"
    for row in parsed_tor_df.itertuples()
)

print(full_tor_context[:2000])
print(f"\nTotal characters: {len(full_tor_context)}")

[PID:16] [SECTION:9. Annexes] 9 List of Annex es
[PID:17] [SECTION:9. Annexes] Annex: 1 Assessment Grid
[PID:18] [SECTION:5. Deliverables] 2 Content and structure of the report
[PID:19] [SECTION:(unknown)] 3 Reference documents
[PID:20] [SECTION:(unknown)] Prepared by: WEHU | Document Version: 4 | Template Version: - | Date: 15. 03.2024
[PID:21] [SECTION:1. Purpose of the Evaluation] Purpose of the Evaluation
[PID:22] [SECTION:1. Purpose of the Evaluation] The S TSM Programme is scheduled to end in December 2024. SECO, as the donor of this program me, commissions an external evaluation in the last year of implementation with the following main objectives:
[PID:23] [SECTION:1. Purpose of the Evaluation] S trategically evaluate the relevance and effectiveness of the design and programme set-up for the implementation of [PERSON]
[PID:24] [SECTION:1. Purpose of the Evaluation] Learn about the success factors and identify m ain bottlenecks for the achievement of project results.
[PID:25] [S

## 4.2.2 Classification of Fail Signals
This step is the core of the ToR Assistant, where the **system checks whether activated fail signals are present in the draft ToR** using an LLM. For each checklist item, the model analyses the full document and classifies the fail signal as **“TRIGGERED”, “NOT_TRIGGERED”, or “UNCERTAIN”**, while also providing supporting paragraph references and a short rationale. The results are returned in a structured JSON format to improve reliability and ensure that every fail signal produces a valid output.

First, we define the prompt for the LLM.

In [ ]:
# Prompt builder and fail-signal checker

load_dotenv(ENV_PATH)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def check_fail_signal(row, full_tor_context, model="gpt-5.2"):
    system_prompt = """
Role:
- You are an evaluation expert reviewing a draft Terms of Reference (ToR).

Task:
- Assess whether the fail signal is triggered anywhere in the full ToR.
- Use the full ToR, not only the named section.
- Be conservative and evidence-based.
- If evidence is mixed or contradictory, return UNCERTAIN.
- DO NOT directly mention paragraph IDs (e.g., PID 12) in the rationale.

Return ONLY valid JSON:
{
  "fail_signal_id": "...",
  "decision": "TRIGGERED" | "NOT_TRIGGERED" | "UNCERTAIN",
  "evidence_paragraph_ids": [1, 2],
  "rationale": "short explanation"
}
""".strip()

    user_prompt = f"""
FAIL SIGNAL ID: {row.fail_signal_id}
PRIMARY SECTION: {row.chapter_text}
FAIL SIGNAL: {row.fail_signal}

FULL TOR:
{full_tor_context}
""".strip()

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    raw = response.output_text.strip()

    try:
        return json.loads(raw)
    except Exception:
        return {
            "fail_signal_id": row.fail_signal_id,
            "decision": "UNCERTAIN",
            "evidence_paragraph_ids": [],
            "rationale": f"Could not parse model output: {raw[:300]}"
        }

We then **check all activated fail signals against the full ToR**, collects the model’s decisions, and links the results to the relevant evidence paragraphs. We display the final assessment table with the decisions, evidence, and explanations.

In [ ]:
# Run checks for all activated fail signals
results = []
for i, row in enumerate(fail_signals_df.itertuples(), start=1):
    print(f"Checking {i}/{len(fail_signals_df)}", end="\r")
    results.append(check_fail_signal(row, full_tor_context))

results_df = pd.DataFrame(results)

# Ensure same type for merge key
results_df["fail_signal_id"] = results_df["fail_signal_id"].astype(str)
fail_signals_df["fail_signal_id"] = fail_signals_df["fail_signal_id"].astype(str)

# Add fail signal text
results_df = results_df.merge(
    fail_signals_df[["fail_signal_id", "chapter_text", "fail_signal"]],
    on="fail_signal_id",
    how="left"
)

# Paragraph lookup
paragraph_lookup = parsed_tor_df.set_index("paragraph_id")["paragraph_text"].to_dict()

# Map evidence IDs -> text
def get_evidence_text(paragraph_ids):
    if not isinstance(paragraph_ids, list):
        return ""
    return "\n\n".join(
        f"[PID:{pid}] {paragraph_lookup.get(pid, '[Missing]')}"
        for pid in paragraph_ids
    )

results_df["evidence_text"] = results_df["evidence_paragraph_ids"].apply(get_evidence_text)

# Display
display(results_df[[
    "fail_signal_id",
    "chapter_text",
    "fail_signal",
    "decision",
    "evidence_paragraph_ids",
    "evidence_text",
    "rationale"
]])

,fail_signal_id,chapter_text,fail_signal,decision,evidence_paragraph_ids,evidence_text,rationale
0,1.0,1. Purpose of the Evaluation,Multiple evaluation purposes are mentioned but no primary purpose or equal weighting is explicitly stated.,NOT_TRIGGERED,"[22, 23, 24, 25, 26]","[PID:22] The S TSM Programme is scheduled to end in December 2024. SECO, as the donor of this program me, commissions an external evaluation in the last year of implementation with the following main objectives:\n\n[PID:23] S trategically evaluate the relevance and effectiveness of the design and programme set-up for the implementation of [PERSON]\n\n[PID:24] Learn about the success factors and identify m ain bottlenecks for the achievement of project results.\n\n[PID:25] Based on these insights, conduct a c ritical review of Phase I a nd make r ecommendations on the future direction of the project.\n\n[PID:26] The insights f rom the evaluation will be used for designing the potential follow-up phase, which is schedule d to start in Q12025.","While multiple objectives are listed (strategic evaluation, learning, and recommendations), the ToR explicitly frames them as the “main objectives” of the evaluation and indicates an overarching use (informing design of a potential follow-up phase). This provides an implied primary orientation rather than presenting multiple unrelated purposes with no prioritization."
1,2.0,1. Purpose of the Evaluation,No evaluation purpose is stated.,NOT_TRIGGERED,"[21, 22, 23, 24, 25, 26]","[PID:21] Purpose of the Evaluation\n\n[PID:22] The S TSM Programme is scheduled to end in December 2024. SECO, as the donor of this program me, commissions an external evaluation in the last year of implementation with the following main objectives:\n\n[PID:23] S trategically evaluate the relevance and effectiveness of the design and programme set-up for the implementation of [PERSON]\n\n[PID:24] Learn about the success factors and identify m ain bottlenecks for the achievement of project results.\n\n[PID:25] Based on these insights, conduct a c ritical review of Phase I a nd make r ecommendations on the future direction of the project.\n\n[PID:26] The insights f rom the evaluation will be used for designing the potential follow-up phase, which is schedule d to start in Q12025.","The ToR clearly states the purpose/objectives of the evaluation: it is an external evaluation commissioned in the last implementation year to assess relevance and effectiveness, identify success factors and bottlenecks, critically review Phase I, and produce recommendations to inform the design of a potential follow-up phase. Therefore, an evaluation purpose is stated."
2,4.0,1. Purpose of the Evaluation,The ToR does not specify which decisions or actions will be informed by the evaluation findings.,NOT_TRIGGERED,"[25, 26, 70, 183]","[PID:25] Based on these insights, conduct a c ritical review of Phase I a nd make r ecommendations on the future direction of the project.\n\n[PID:26] The insights f rom the evaluation will be used for designing the potential follow-up phase, which is schedule d to start in Q12025.\n\n[PID:70] SECO will draft a management response to the F inal E valuation R eport.\n\n[PID:183] Recommendations for the second phase (max. 4 pages):","The ToR specifies intended uses of the evaluation findings, linking them to concrete forward-looking actions/decisions. It states the evaluation will produce recommendations on the future direction of the project and that insights will be used to design a potential follow-up phase. It also indicates SECO will produce a management response to the final report, implying follow-up actions based on findings, and the report structure explicitly includes recommendations for a second phase."
3,5.0,1. Purpose of the Evaluation,The intended users or audiences of the evaluation are not mentioned.,NOT_TRIGGERED,"[22, 26, 69, 70, 75, 76]","[PID:22] The S TSM Programme is scheduled to end in December 2024. SECO, as the donor 

## 4.2.3	Generating Revision Guidance and Text Suggestions
This step **generates guidance for fail signals that were classified as “TRIGGERED”**. Depending on the type of issue and the available evidence in the ToR, the system **either provides predefined revision instructions or additionally generates a grounded text suggestion**. Suggestions are only created when sufficient evidence is available, which helps reduce unreliable or hallucinated outputs.

To start, we prepare the triggered fail signals for the revision step by linking them with the corresponding fix instructions and checking whether enough evidence exists to generate a grounded suggestion.

In [ ]:
# Check whether enough evidence exists to generate a grounded suggestion
def has_sufficient_evidence(row, min_paragraphs=1, min_chars=120):
    pids = row.get("evidence_paragraph_ids", [])
    txt = str(row.get("evidence_text", "") or "").strip()

    return (
        isinstance(pids, list)
        and len(pids) >= min_paragraphs
        and len(txt) >= min_chars
    )

# Start from the classification results
fix_input_df = results_df.copy()

# Keep only fail signals classified as TRIGGERED
fix_input_df = fix_input_df[fix_input_df["decision"] == "TRIGGERED"].copy()

# Ensure IDs have consistent format before merging
fix_input_df["fail_signal_id"] = fix_input_df["fail_signal_id"].astype(str)
fail_signals_df["fail_signal_id"] = fail_signals_df["fail_signal_id"].astype(str)

# Add fix instructions and metadata from the checklist
fix_input_df = fix_input_df.merge(
    fail_signals_df[["fail_signal_id", "fix_pattern", "fix_mode", "importance"]],
    on="fail_signal_id",
    how="left"
)

# Use instruction-only mode if no mode is defined
fix_input_df["fix_mode"] = fix_input_df["fix_mode"].fillna("instruction_only")

# Check whether enough evidence exists for generating suggestions
fix_input_df["sufficient_evidence"] = fix_input_df.apply(has_sufficient_evidence, axis=1)

# Display the information for the generated fix candidates
display(fix_input_df[[
    "fail_signal_id",
    "fail_signal",
    "fix_mode",
    "importance",
    "sufficient_evidence",
    "fix_pattern"
]])

,fail_signal_id,fail_signal,fix_mode,importance,sufficient_evidence,fix_pattern
0,17.0,No summary of the projectÕs implementation structure is provided.,instruction_only,medium,True,Summarise the projectÕs implementation structure clearly and concisely.
1,19.0,"The ToR does not include, annex, or reference a Theory of Change, nor does it explicitly state that one does not exist. Mentioning a Theory of Change only in the SECO assessment grid, as an evaluation deliverable or in the standard format for evaluation reports is insufficient.",instruction_only,high,True,"Clarify whether a Theory of Change exists and, if so, include or reference it in the ToR."
2,20.0,"The ToR neither includes, annexes, nor references a logframe or results framework, and does not explicitly state that they do not exist. Mentioning a logframe or results framework only in the SECO assessment grid, as an evaluation deliverable or in the standard format for evaluation reports is insufficient.",instruction_only,high,True,"Clarify whether a logframe or results framework exists and, if so, include or reference it in the ToR."
3,22.0,"Data, which is already available to the evaluators is not described, or fewer than three of the following aspects are addressed: sources, formats, coverage, quality and limitations.",instruction_only,high,True,"Summarise the available data the evaluator can use, covering at least three aspects: sources, formats, coverage, quality, or limitations."
4,27.0,"The ToR contains more than 10 evaluation questions in total, including subquestions.",instruction_only,high,True,"Review and, where possible, reduce the number of evaluation questions to a manageable set. At SECO-WE, a maximum of 10 questions is typically recommended."
5,36.0,"All six OECD-DAC criteria (relevance, coherence, effectiveness, efficiency, impact, sustainability) are included.",instruction_only,high,True,Specify and prioritise the most relevant DAC criteria. Do not include all criteria unless necessary.
6,40.0,"Evaluation questions are suggestive, biased, or assume a particular outcome.",instruction_only,high,True,Formulate neutral evaluation questions. Avoid suggestive or biased wording.
7,41.0,Evaluation questions combine multiple issues in one question (multi-part questions).,instruction_only,high,True,"Split evaluation questions into single, focused questions. Avoid combining multiple issues."
8,42.0,The evaluation questions are overlapping and partially address the same issue.,instruction_only,high,True,Ensure evaluation questions are distinct and do not overlap.
9,45.0,"Evaluation questions are misaligned with OECD-DAC criteria, incorrectly categorized, or assigned in a way that contradicts the conceptual definitions of the criteria (e.g., questions on results or contribution placed under relevance instead of effectiveness or impact).",instruction_only,high,True,Align evaluation questions with the correct OECD-DAC criteria. Ensure consistent and accurate categorisation.


Next, the system generates revision guidance for each triggered fail signal by returning predefined instructions and, where possible, an additional grounded text suggestion based on the ToR content.

In [ ]:
# Generate guidance
def generate_guidance(row, full_tor_context, model="gpt-5.2"):
    base_instruction = str(row.get("fix_pattern", "")).strip()
    mode = row.get("fix_mode") or "instruction_only"

    if mode == "instruction_only":
        return {
            "fail_signal_id": row["fail_signal_id"],
            "target_section": row.get("chapter_text", ""),
            "instruction_text": base_instruction,
            "suggested_text": ""
        }

    if mode == "instruction_plus_suggestion" and row.get("sufficient_evidence"):
        system_prompt = """
You support improving Terms of Reference (ToR).

Task:
- Keep the instruction as given.
- Provide a short, concrete suggestion based ONLY on the ToR.
- Do not invent missing information.
- If no good suggestion is possible, return an empty string.

Return JSON:
{
  "suggested_text": "..."
}
""".strip()

        user_prompt = f"""
INSTRUCTION:
{base_instruction}

FAIL SIGNAL:
{row['fail_signal']}

EVIDENCE:
{row['evidence_text']}

FULL TOR:
{full_tor_context}
""".strip()

        response = client.responses.create(
            model=model,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )

        try:
            out = json.loads(response.output_text.strip())
            suggestion = str(out.get("suggested_text", "")).strip()
        except Exception:
            suggestion = ""
    else:
        suggestion = ""

    return {
        "fail_signal_id": row["fail_signal_id"],
        "target_section": row.get("chapter_text", ""),
        "instruction_text": base_instruction,
        "suggested_text": suggestion
    }

Next, we run the guidance generation step for all triggered fail signals and store the resulting instructions and suggestions in a structured dataframe.

In [ ]:
# Run guidance generation
fix_results = []
for i, row in enumerate(fix_input_df.itertuples(index=False), start=1):
    print(f"Generating guidance {i}/{len(fix_input_df)}", end="\r")
    fix_results.append(generate_guidance(row._asdict(), full_tor_context))

fix_results_df = pd.DataFrame(fix_results)

display(fix_results_df)

,fail_signal_id,target_section,instruction_text,suggested_text
0,17.0,2. Description of the Intervention,Summarise the projectÕs implementation structure clearly and concisely.,
1,19.0,2. Description of the Intervention,"Clarify whether a Theory of Change exists and, if so, include or reference it in the ToR.",
2,20.0,2. Description of the Intervention,"Clarify whether a logframe or results framework exists and, if so, include or reference it in the ToR.",
3,22.0,2. Description of the Intervention,"Summarise the available data the evaluator can use, covering at least three aspects: sources, formats, coverage, quality, or limitations.",
4,27.0,3. Scope and Focus of the Evaluation Evaluation Questions,"Review and, where possible, reduce the number of evaluation questions to a manageable set. At SECO-WE, a maximum of 10 questions is typically recommended.",
5,36.0,3. Scope and Focus of the Evaluation Evaluation Questions,Specify and prioritise the most relevant DAC criteria. Do not include all criteria unless necessary.,
6,40.0,3. Scope and Focus of the Evaluation Evaluation Questions,Formulate neutral evaluation questions. Avoid suggestive or biased wording.,
7,41.0,3. Scope and Focus of the Evaluation Evaluation Questions,"Split evaluation questions into single, focused questions. Avoid combining multiple issues.",
8,42.0,3. Scope and Focus of the Evaluation Evaluation Questions,Ensure evaluation questions are distinct and do not overlap.,
9,45.0,3. Scope and Focus of the Evaluation Evaluation Questions,Align evaluation questions with the correct OECD-DAC criteria. Ensure consistent and accurate categorisation.,


Lastly, we merge the generated guidance back into the main results table and add the importance level of each fail signal. This creates a final structured dataset that combines the classification results, revision guidance, and prioritisation information.

In [ ]:
# Merge guidance results
fix_cols = [
    "fail_signal_id",
    "target_section",
    "instruction_text",
    "suggested_text"
]

results_df["fail_signal_id"] = results_df["fail_signal_id"].astype(str)
fix_results_df["fail_signal_id"] = fix_results_df["fail_signal_id"].astype(str)
fail_signals_df["fail_signal_id"] = fail_signals_df["fail_signal_id"].astype(str)

results_with_fixes_df = results_df.merge(
    fix_results_df[fix_cols],
    on="fail_signal_id",
    how="left"
)

for col in fix_cols[1:]:
    results_with_fixes_df[col] = results_with_fixes_df[col].fillna("")

results_with_fixes_df = results_with_fixes_df.merge(
    fail_signals_df[["fail_signal_id", "importance"]],
    on="fail_signal_id",
    how="left"
)

importance_order = {"high": 0, "medium": 1, "low": 2}
results_with_fixes_df["importance_rank"] = (
    results_with_fixes_df["importance"].map(importance_order).fillna(3)
)

report_fixes_df = results_with_fixes_df.copy()

## 4.2.4 Generating ToR Review Report
The last step generates a ToR Review Report that combines the classification results, rationales, and revision guidance into a single document. The report highlights the most important triggered fail signals, provides detailed explanations and text suggestions where available, and includes references to the relevant ToR excerpts for transparency. The final output is exported as a Word document.

The model first summarises the detected issues, rationales, and guidance into short, action-oriented review statements for the final report. Since this task is relatively simple, a lighter and cheaper GPT model is sufficient.

In [ ]:
# Generate concise summary items
def generate_summary_item(row, model="gpt-4o-mini"):

    system_prompt = """
You summarise evaluation findings for a report.

Task:
- Create one short, user-facing summary sentence that describes the issue and includes a brief action.

Rules:
- Plain English
- No repetition
- Action-oriented (e.g. "clarify", "align", "define")
- Do NOT repeat full fix patterns
- Keep it very concise (one sentence)

Return JSON:
{
  "summary_text": "..."
}
"""

    user_prompt = f"""
FAIL SIGNAL:
{row['fail_signal']}

RATIONALE:
{row['rationale']}

GUIDANCE:
{row.get('instruction_text', '')}
"""

    response = client.responses.create(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    try:
        out = json.loads(response.output_text.strip())
    except Exception:
        out = {}

    return {
        "fail_signal_id": row["fail_signal_id"],
        "summary_text": str(out.get("summary_text", "")).strip()
    }


summary_input_df = results_with_fixes_df[
    results_with_fixes_df["decision"].isin(["TRIGGERED", "UNCERTAIN"])
].copy()

summary_results = []
for i, row in enumerate(summary_input_df.itertuples(index=False), start=1):
    print(f"Summarising {i}/{len(summary_input_df)}", end="\r")
    summary_results.append(generate_summary_item(row._asdict()))

summary_df = pd.DataFrame(summary_results)

results_with_fixes_df = results_with_fixes_df.merge(
    summary_df,
    on="fail_signal_id",
    how="left"
)

Next, we prepare and export the results into a Excel file containing the detected fail signals, rationales, guidance, suggestions, and supporting evidence.

In [ ]:
# Export review dataframe
importance_order = {"high": 0, "medium": 1, "low": 2}
results_with_fixes_df["importance_rank"] = results_with_fixes_df["importance"].map(importance_order)

export_cols = [
    "fail_signal_id",
    "fail_signal",
    "decision",
    "summary_text",
    "rationale",
    "target_section",
    "instruction_text",
    "suggested_text",
    "evidence_paragraph_ids",
    "evidence_text",
]

review_export_df = results_with_fixes_df.copy()

review_export_df["fail_signal_id"] = (
    pd.to_numeric(review_export_df["fail_signal_id"], errors="coerce")
    .fillna(0)
    .astype(int)
)

review_export_df = (
    review_export_df[export_cols]
    .sort_values("fail_signal_id")
    .reset_index(drop=True)
)

excel_path = f"{PROJECT_ROOT}/Output/ToR_Review_Data.xlsx"
review_export_df.to_excel(excel_path, index=False)

print(excel_path)

/content/drive/MyDrive/HSLU/Masters_Thesis/Output/ToR_Review_Data.xlsx


 We extract the project name from the ToR for the review report. To achive this, we extracted the raw text from the original .docx file and used rule-based logic plus a (lightweight) LLM to identify the actual project name from the ToR cover page.

In [ ]:
def extract_docx_xml_text(docx_path):
    NS = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}

    with zipfile.ZipFile(docx_path) as z:
        root = ET.fromstring(z.read("word/document.xml"))

    lines = []

    for p in root.findall(".//w:p", NS):
        texts = [t.text for t in p.findall(".//w:t", NS) if t.text]

        line = " ".join(texts)
        line = re.sub(r"\s+", " ", line).strip()

        if line:
            lines.append(line)

    return "\n".join(lines)


def detect_project_name_from_docx(docx_path, model="gpt-4o-mini"):

    cover_text = extract_docx_xml_text(docx_path)

    lines = [
        re.sub(r"\s+", " ", x).strip()
        for x in cover_text.splitlines()
        if x.strip()
    ]

    # Strong rule:
    # External Evaluation / of / PROJECT TITLE / in COUNTRY
    for i, line in enumerate(lines):

        if line.lower() == "of" and i + 1 < len(lines):

            candidate = re.sub(r"\s+", " ", lines[i + 1]).strip()

            # Remove trailing country references if they leaked in
            candidate = re.sub(
                r"\s+in\s+[A-Z][A-Za-z\s]+$",
                "",
                candidate
            ).strip()

            if (
                len(candidate) > 5
                and "evaluation" not in candidate.lower()
                and "terms of reference" not in candidate.lower()
            ):
                return candidate

    # Regex fallback
    m = re.search(
        r"External Evaluation\s+of\s+(.+?)\s+in\s+[A-Z][A-Za-z\s]+",
        cover_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if m:
        candidate = re.sub(r"\s+", " ", m.group(1)).strip()
        return candidate

    # LLM fallback
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "system",
                "content": (
                    "Extract the exact project/programme/intervention name from a ToR cover page. "
                    "Return ONLY valid JSON: "
                    "{\"project_name\":\"...\"}. "
                    "Return only the project name, not the file name, "
                    "country, SAP, ACTA NOVA ID, or 'External Evaluation'."
                )
            },
            {
                "role": "user",
                "content": cover_text[:6000]
            }
        ]
    )

    try:
        out = json.loads(response.output_text.strip())

        candidate = str(out.get("project_name", "")).strip()
        candidate = re.sub(r"\s+", " ", candidate)

        return candidate

    except Exception:
        return ""


project_title = detect_project_name_from_docx(TOR_INPUT_PATH)

if not project_title:
    project_title = Path(TOR_INPUT_PATH).stem.replace("_", " ")

print("Project title:", project_title)

Project title: Sustainable Tourism Switzerland-Morocco Programme (STSM)


At the end of the pipline, we generate the final Word report (ToR Review Report) by formatting the review results into an overall assessment, document overview, key issues, and detailed findings. The report is then saved as a .docx file.

In [ ]:
# Creating the ToR Review Report
results_with_fixes_df["fail_signal_id"] = (
    pd.to_numeric(results_with_fixes_df["fail_signal_id"], errors="coerce")
    .fillna(0)
    .astype(int)
)

doc = Document()

# Global font
style = doc.styles["Normal"]
font = style.font
font.name = "Arial"
font.size = Pt(11)
font.color.rgb = RGBColor(0, 0, 0)

# Headings
for level in range(1, 4):
    style = doc.styles[f"Heading {level}"]
    style.font.name = "Arial"
    style.font.size = Pt(12 if level == 1 else 11)
    style.font.color.rgb = RGBColor(0, 0, 0)

# Helpers
def clean_text(x):
    return str(x).strip() if pd.notna(x) else ""

def pct(x, base):
    return f"{round((x / base) * 100)}%" if base > 0 else "0%"

def add_bookmark(paragraph, name):
    tag = OxmlElement("w:bookmarkStart")
    tag.set(qn("w:id"), str(abs(hash(name)) % 100000))
    tag.set(qn("w:name"), name)
    paragraph._p.insert(0, tag)

    end = OxmlElement("w:bookmarkEnd")
    end.set(qn("w:id"), tag.get(qn("w:id")))
    paragraph._p.append(end)

def add_link(paragraph, text, anchor):
    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("w:anchor"), anchor)

    run = OxmlElement("w:r")
    rPr = OxmlElement("w:rPr")

    rStyle = OxmlElement("w:rStyle")
    rStyle.set(qn("w:val"), "Hyperlink")
    rPr.append(rStyle)

    run.append(rPr)

    t = OxmlElement("w:t")
    t.text = text
    run.append(t)

    hyperlink.append(run)
    paragraph._p.append(hyperlink)

# Sorting
sorted_df = results_with_fixes_df.sort_values(["importance_rank", "fail_signal_id"])

# Metadata
file_name = Path(TOR_INPUT_PATH).name

# Header: logo + title
table = doc.add_table(rows=1, cols=2)
table.autofit = True
table.columns[0].width = Cm(6)
table.columns[1].width = Cm(10)

cell_logo = table.rows[0].cells[0]
p_logo = cell_logo.paragraphs[0]
p_logo.alignment = WD_ALIGN_PARAGRAPH.LEFT

run_logo = p_logo.add_run()
run_logo.add_picture(LOGO_PATH, height=Cm(2.0))

cell_title = table.rows[0].cells[1]
p_title = cell_title.paragraphs[0]
p_title.alignment = WD_ALIGN_PARAGRAPH.RIGHT

run = p_title.add_run("ToR Review Report (TRR)")
run.bold = True
run.font.name = "Arial"
run.font.size = Pt(14)

doc.add_paragraph(f"Project reviewed: {project_title}")

doc.add_paragraph(
    f"This report presents an AI-generated review of the evaluation Terms of Reference (ToR) of {project_title}. "
    "It provides an overall assessment and summarises relevant technical background information. "
    "The key issues section is organised by importance and offers concise guidance on how the ToR can be improved. "
    "For additional context, each key issue includes a link to the detailed finding. "
    "Please note that generative AI systems can produce inaccuracies or incomplete information."
)

# Overall assessment
doc.add_heading("Overall assessment", 1)

total = len(results_with_fixes_df)
n_tr = (results_with_fixes_df["decision"] == "TRIGGERED").sum()
n_nt = (results_with_fixes_df["decision"] == "NOT_TRIGGERED").sum()
n_un = (results_with_fixes_df["decision"] == "UNCERTAIN").sum()

# Review scope table
doc.add_heading("Review scope", 2)

scope_table = doc.add_table(rows=1, cols=2)
scope_table.style = "Table Grid"

hdr = scope_table.rows[0].cells
hdr[0].text = "Item"
hdr[1].text = "Value"

scope_rows = [
    ("Project reviewed", project_title),
    ("Reviewed file", file_name),
    ("Paragraphs parsed", str(len(parsed_tor_df))),
    ("Analysed text length", f"{len(full_tor_context):,} characters"),
]

for label, value in scope_rows:
    r = scope_table.add_row().cells
    r[0].text = label
    r[1].text = value

doc.add_paragraph(
    "Note: Technical counts may differ from the visible document layout because headings, tables, "
    "and formatting are converted into structured text during parsing."
)

# Review results table
doc.add_heading("Review results", 2)

results_table = doc.add_table(rows=1, cols=3)
results_table.style = "Table Grid"

hdr = results_table.rows[0].cells
hdr[0].text = "Category"
hdr[1].text = "Count"
hdr[2].text = "Share"

result_rows = [
    ("Fail signals triggered", n_tr, pct(n_tr, total)),
    ("Uncertain", n_un, pct(n_un, total)),
    ("Fail signals not triggered", n_nt, pct(n_nt, total)),
    ("Total checks", total, "100%"),
]

for label, count, share in result_rows:
    r = results_table.add_row().cells
    r[0].text = label
    r[1].text = str(count)
    r[2].text = share

    if label == "Total checks":
        for cell in r:
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.bold = True

# Key issues
doc.add_heading("Key issues", 1)

def render_group(title, df):
    if len(df) == 0:
        return

    doc.add_heading(title, 2)

    for _, row in df.iterrows():
        text = clean_text(row.get("summary_text", ""))
        if not text:
            continue

        p = doc.add_paragraph()
        p.add_run("→ " + text + " ")
        add_link(p, "Click here for details.", f"fs_{row['fail_signal_id']}")

render_group("High importance", sorted_df[(sorted_df.decision == "TRIGGERED") & (sorted_df.importance == "high")])
render_group("Medium importance", sorted_df[(sorted_df.decision == "TRIGGERED") & (sorted_df.importance == "medium")])
render_group("Low importance", sorted_df[(sorted_df.decision == "TRIGGERED") & (sorted_df.importance == "low")])
render_group("Uncertain", sorted_df[sorted_df.decision == "UNCERTAIN"])

# Detailed findings
doc.add_heading("Detailed findings", 1)

detail_df = sorted_df[sorted_df.decision.isin(["TRIGGERED", "UNCERTAIN"])]

for _, row in detail_df.iterrows():

    sep = doc.add_paragraph()
    run = sep.add_run("—" * 50)
    run.font.size = Pt(8)

    key_issue = clean_text(row.get("summary_text", ""))
    if not key_issue:
        key_issue = clean_text(row.get("fail_signal", ""))

    p = doc.add_paragraph()
    run = p.add_run(key_issue)
    run.bold = True
    run.font.size = Pt(11)

    add_bookmark(p, f"fs_{row['fail_signal_id']}")

    p = doc.add_paragraph()
    p.add_run("Status: ").bold = True
    p.add_run(clean_text(row["decision"]))

    p = doc.add_paragraph()
    p.add_run("Fail signal: ").bold = True
    p.add_run(clean_text(row["fail_signal"]))

    p = doc.add_paragraph()
    p.add_run("Rationale: ").bold = True
    p.add_run(clean_text(row["rationale"]))

    if row.get("instruction_text"):
        p = doc.add_paragraph()
        p.add_run("Assessment: ").bold = True
        p.add_run(clean_text(row["instruction_text"]))

    if row.get("suggested_text"):
        p = doc.add_paragraph()
        p.add_run("Text suggestion: ").bold = True
        p.add_run(clean_text(row["suggested_text"]))

    doc.add_paragraph()

# Save
output_path = f"{PROJECT_ROOT}/Output/ToR_Review_Report.docx"
doc.save(output_path)

print(output_path)

/content/drive/MyDrive/HSLU/Masters_Thesis/Output/ToR_Review_Report.docx
